[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/electromagnetism/magnetic_bottle/magnetic_bottle.ipynb)

# The Magnetic Bottle

This example computes the relativistic trajectories of charged particles between two coaxial current loops that form a magnetic bottle. A charged particle in a magnetic field gyrates around the field lines while drifting along them at constant speed, because a purely magnetic force does no work. Where the field varies slowly across one gyro-orbit, the orbital magnetic moment stays nearly constant, so a particle drifting into the converging field near a coil trades axial drift for faster gyration until its axial motion reverses at a magnetic mirror. Particles launched at a steep enough pitch bounce repeatedly between the two coils, whereas particles launched within a narrow pitch angle around the axis pass through the throat before turning back; that escape boundary is called the loss cone.

In spacetime, events, four-velocities, spatial separations, and wire current elements are modelled as intrinsic antivectors, while the magnetic field is modelled as an extrinsic bivector built directly from each wire segment and separation as `currents.commutator(separation)`. Pairing the field with the four-velocity via `magnetic & velocity` gives the Lorentz four-force covector, whose spacetime dual `velocity.commutator(magnetic)` is the proper-time rate at which the field turns the velocity, and advancing each step with the rotor `(magnetic * (proper_step / 2)).exp() >> velocity` preserves unit spacetime norm at any step size.

In Gibbs–Heaviside vector notation, Biot–Savart forms an axial magnetic vector by the right-hand cross product $\mathbf{B}(\mathbf{x}) = \sum_k I\,\Delta\mathbf{l}_k \times (\mathbf{x} - \mathbf{x}_k)/|\mathbf{x} - \mathbf{x}_k|^3$, relativistic motion updates $\gamma\mathbf{v}$ with $\gamma = (1 - |\mathbf{v}|^2)^{-1/2}$ through $d(\gamma\mathbf{v})/dt = (q/m)\,\mathbf{v} \times \mathbf{B}$ via a Boris half-tangent cross-product step, and the first adiabatic invariant reads $\mu = \gamma^2 |\mathbf{v} \times \mathbf{B}|^2 / (2|\mathbf{B}|^3)$ in units of $q/m = 1$.

In [ ]:
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
from collections.abc import Iterator

import numpy as np

from examples.electromagnetism.magnetic_bottle import render
from examples.util.animation import show_states
from numga import Algebra, NumpyContext, stack

np.set_printoptions(precision=3, suppress=True)

ga = Algebra("x+y+z+t-")
mv = NumpyContext(ga).multivector

Scalar = ga.gatype.scalar()                                                   # Scalar+
Field = ga.gatype.bivector()                                                  # Bivector+
Antivector = ga.gatype.antivector()                                           # Antivector-

# Dualizing the spacetime basis planes gives laboratory time and orthonormal spatial axes.
time = -mv.t.dual()                                                           # [] Antivector-
x_axis, y_axis, z_axis = stack([mv.x, mv.y, mv.z]).dual()                     # [] Antivector-, Antivector-, Antivector-

## Coils and their magnetic field

Each circular coil is modelled as a ring of straight current segments around the bottle axis. By the Biot–Savart law, each segment contributes the spacetime bivector `currents.commutator(separation)` divided by the cube of the spatial distance, building the magnetic field plane directly from the wire geometry without a right-hand rule. Along the central axis, the ratio of the midplane field minimum to the throat maximum sets the pitch angle of the loss cone.

In [ ]:
coil_radius = 1.0
coil_heights = np.array([-2.0, 2.0])
n_pieces = 48
coil_current = 1.78

n_axis = 401
view_half_length = 3.0
view_half_width = 1.2
n_grid = 120

In [ ]:
def coils(radius: float, heights: np.ndarray, current: float, n_pieces: int) -> tuple[Antivector, Antivector]:
    """Circular coils of `radius` and `current` at `heights` (`[n_coils]`) discretize into straight
    current elements and their midpoints (`[n_coils * n_pieces] Antivector-`)."""
    n_coils = len(heights)
    angles = np.linspace(0, 2 * np.pi, n_pieces + 1)
    turns = (mv.xy * (-angles / 2)).exp()                                     # [n_pieces + 1] Rotor+
    rim = turns >> (x_axis * radius)                                          # [n_pieces + 1] Antivector-
    wire = rim + z_axis * heights[:, None]                                    # [n_coils, n_pieces + 1] Antivector-
    currents = current * (wire[:, 1:] - wire[:, :-1]).reshape(n_coils * n_pieces)   # [n_coils * n_pieces] Antivector-
    midpoints = ((wire[:, 1:] + wire[:, :-1]) / 2).reshape(n_coils * n_pieces)      # [n_coils * n_pieces] Antivector-
    return currents, midpoints


def field(events: Antivector, currents: Antivector, midpoints: Antivector) -> Field:
    """Steady wire `currents` and `midpoints` produce the Biot–Savart magnetic field (`[...] Field+`)
    at `events` (`[...] Antivector-`)."""
    # Intersecting each event's laboratory worldline with the constant-time hyperplane reads its spatial position.
    position = mv.t ^ (events & time)                                         # [...] Antivector-
    separation = position[..., None] - midpoints                              # [...] Antivector-
    distance = separation.antinorm()                                          # [...] Scalar+
    # The commutator of two spatial antivectors spans their extrinsic magnetic plane.
    planes = currents.commutator(separation)                                  # [...] Field+
    return (planes / (distance * distance.squared())).sum(axis=-1)            # [...] Field+


currents, midpoints = coils(coil_radius, coil_heights, coil_current, n_pieces)
axis_samples = z_axis * np.linspace(coil_heights[0], coil_heights[-1], n_axis)   # [n_axis] Antivector-
axis_strengths = field(axis_samples, currents, midpoints).norm()              # [n_axis] Scalar+
minimum_strength = axis_strengths.min()                                       # [] Scalar+
maximum_strength = axis_strengths.max()                                       # [] Scalar+
loss_cone = (minimum_strength / maximum_strength).square_root().arcsin()      # [] Scalar+
loss_cone_degrees = loss_cone * (180 / np.pi)                                 # [] Scalar+

axial_points = z_axis * np.linspace(-view_half_length, view_half_length, 2 * n_grid)   # [2 * n_grid] Antivector-
radial_offsets = x_axis * np.linspace(-view_half_width, view_half_width, n_grid)[:, None]   # [n_grid, 1] Antivector-
points = radial_offsets + axial_points                                        # [n_grid, 2 * n_grid] Antivector-
strength = field(points, currents, midpoints).norm()                          # [n_grid, 2 * n_grid] Scalar+

print(f"field on axis from {minimum_strength.to_array():.3f} to {maximum_strength.to_array():.3f}, loss cone {loss_cone_degrees.to_array():.1f} degrees")

## Particle trajectories in the bottle

Particles launch from the midplane at increasing radial offsets and pitch angles to the bottle axis. Shallow trajectories inside the loss cone pass straight through the coil throats, whereas steeper trajectories mirror before reaching the coils and precess slowly around the axis as the off-axis field curves.

In [ ]:
launch_speed = 0.3
n_particles = 10
launch_pitches = np.linspace(15, 80, n_particles)
launch_offsets = np.linspace(0, 0.6, n_particles)

proper_step = 0.1
n_steps = 1000
n_steps_per_frame = 8
frame_duration_ms = 40

In [ ]:
def step(
    event: Antivector,
    velocity: Antivector,
    currents: Antivector,
    midpoints: Antivector,
    proper_step: float,
) -> tuple[Antivector, Antivector]:
    """Leapfrog proper-time integration advances `event` and turns `velocity` (`[...] Antivector-`)
    through `proper_step` at unit charge-to-mass ratio."""
    midpoint_event = event + velocity * (proper_step / 2)                     # [...] Antivector-
    magnetic = field(midpoint_event, currents, midpoints)                     # [...] Field+
    # Exponentiating half the field step integrates velocity.commutator(magnetic) at exact unit speed.
    turn = (magnetic * (proper_step / 2)).exp()                               # [...] Rotor+
    velocity = turn >> velocity                                               # [...] Antivector-
    return midpoint_event + velocity * (proper_step / 2), velocity


def evolve(
    event: Antivector,
    velocity: Antivector,
    currents: Antivector,
    midpoints: Antivector,
    proper_step: float,
    n_steps: int,
) -> Iterator[tuple[Antivector, Antivector]]:
    """Yields the particle events and four-velocities (`[...] Antivector-`) across `n_steps`
    proper-time steps."""
    for _ in range(n_steps):
        yield event, velocity
        event, velocity = step(event, velocity, currents, midpoints, proper_step)


tilts = (mv.zy * (-np.radians(launch_pitches) / 2)).exp()                     # [n_particles] Rotor+
directions = tilts >> z_axis                                                  # [n_particles] Antivector-
launch_velocities = (time + launch_speed * directions) / np.sqrt(1 - launch_speed**2)   # [n_particles] Antivector-
states = show_states(
    evolve(x_axis * launch_offsets, launch_velocities, currents, midpoints, proper_step, n_steps),
    render.animate_flights,
    launch_pitches, points, strength, coil_radius, coil_heights, n_steps_per_frame,
    name="magnetic_bottle",
    duration_ms=frame_duration_ms,
    n_frames=n_steps // n_steps_per_frame,
);

The blue meridional sheet shows the logarithm of the magnetic field strength between the two amber coils, darkening into the throats. Deep purple trails mark the shallowest launch pitches escaping past the coils, while warmer magenta-to-yellow trails mark steeper pitches mirroring closer to the midplane as their gyro-centres drift around the axis.

## Adiabatic invariance of the magnetic moment

For a purely magnetic field, the Lorentz four-force covector `magnetic & velocity` lies orthogonal to laboratory time and has magnitude equal to the perpendicular momentum times the field strength, so squaring it and dividing by twice the cubed field strength gives the orbital magnetic moment. Evaluated at the instantaneous particle position, that ratio oscillates within each gyro-orbit wherever the field varies across the circle, whereas its sliding average over one gyro-period stays nearly constant across repeated mirror bounces for every trapped particle and departs once an escaping particle leaves the bottle.

In [ ]:
n_turn = 31

In [ ]:
def moment(velocity: Antivector, magnetic: Field) -> Scalar:
    """The Lorentz four-force covector `magnetic & velocity` determines the orbital magnetic moment
    (`[...] Scalar+`) at unit charge-to-mass ratio."""
    force = magnetic & velocity                                               # [...] Vector+
    strength = magnetic.scalar_norm_squared().square_root()                   # [...] Scalar+
    # For a purely magnetic field, the spatial force covector squares to perpendicular momentum squared times field squared.
    return force.squared() / (2 * strength * strength.squared())              # [...] Scalar+


events, velocities = (stack(items) for items in zip(*states))                 # [n_steps, n_particles] Antivector-, Antivector-
magnetic = field(events, currents, midpoints)                                 # [n_steps, n_particles] Field+
moments = moment(velocities, magnetic)                                        # [n_steps, n_particles] Scalar+
cumulative_moments = moments.cumsum(0)                                        # [n_steps, n_particles] Scalar+
means = (cumulative_moments[n_turn:] - cumulative_moments[:-n_turn]) / n_turn   # [n_steps - n_turn, n_particles] Scalar+
moment_drift = (means / means[0] - 1).abs().max(axis=0)                       # [n_particles] Scalar+
escape_mask = (mv.z & events[-1]).abs() > coil_heights[-1]

print("launch pitch (degrees):       ", launch_pitches)
print("outside bottle at end:        ", escape_mask)
print("peak relative moment drift:   ", moment_drift.to_array())

In [ ]:
# checks
lorentz_factor = 1 / np.sqrt(1 - launch_speed**2)
assert ((velocities.commutator(magnetic).dual() - (magnetic & velocities)).norm() < 1e-8).all()
assert (((mv.t & velocities) - lorentz_factor).abs() < 1e-8).all()
assert np.array_equal(escape_mask, np.radians(launch_pitches) < loss_cone)
assert (moment_drift[~escape_mask] < 0.2).all()
assert (moment_drift[escape_mask] > 1.0).all()